## Genre

### Which tools did you use from each of the 3 categories of Visual Narrative (Figure 7 in Segal and Heer). Why?

The initial map with the movies in time uses a consistent visual platform with a timebar. We use continuity editing to show the shift in density.

The maps section starts with an establishing shot, using continuity editing to create a seamless flow from density to focal points. We use a consistent visual platform ending with only certain features being highlighted (feature distinction).

The movie emotional range visualisation uses feature distinction to highlight various movies and animated transitions.

The genre plot uses a short animated transition in the beginning.

The movie scores uses feature distinction to highlight negative/positive scores. Through an animated transition this changes from dots to a histogram.

The next histograms use object continuity (keeping the histogram style across visualisations). They highlight different success metrics, thus using feature distinction. They use short animations when scrolling.

The final visualisation uses the familiar object (the same SF map that has been shown across the website). The user can zoom, move through the map (motion), have a close-up of the movie information.


### Which tools did you use from each of the 3 categories of Narrative Structure (Figure 7 in Segal and Heer). Why?

We wanted to use the Martini Glass Structure for our website, starting with visualizations that require little to no interaction from the user (besides scrolling) and ending with a fully interactive visualization where the user can explore the data on his own.

The ordering is linear. For interactivity, we use in the final map hover highlighting, filtering and selection. As for the messaging, we used headlines, annotations, multi-messaging (same map, different metrics), comment repetition (inside the text through the maps presentation we reference the two density peaks mentioned in a text box above), introductory text and a summary.

## Visualizations

### San Francisco Movies - Maps
This section loads the cleaned movie dataset, creates a unique-title version of the data and exports the Folium maps used for the visual narrative.

In [21]:
# Core imports
from pathlib import Path

import numpy as np
import pandas as pd
import folium
from folium.plugins import HeatMap

# Output folder for generated HTML maps
OUTPUT_DIR = Path("maps_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [22]:
# Load data
DATA_PATH = Path("sf_movies_cleaned.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find sf_movies_cleaned.csv. Place it in the same folder as this notebook "
        "or update DATA_PATH to the correct location."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns.")
df.head()

Loaded 2,244 rows and 30 columns.


,Title,Locations,Fun facts,Production company,Distributor,Director,Writer,Actor 1,Actor 2,Actor 3,...,Certificates,Runtime,Awards,Imdb_rating,Imdb_votes,Metascore,Box_office,Poster,Rt_score,Metacritic_score
0,Milk,El Camino Del Mar,NaN,Focus Features,Focus Features,Gus Van Sant,Dustin Lance Black,Sean Penn,Emile Hirsch,NaN,...,R,128.0,Won 2 Oscars. 66 wins & 146 nominations total,7.5,184363.0,83.0,"$31,841,299",https://m.media-amazon.com/images/M/MV5BOWVmZW...,93%,83/100
1,Chance,Coit Tower,NaN,TVM Productions Inc.,Hulu,Rozann Dawson,Alexandra Cunningham,Hugh Laurie,Greta Lee,Ethan Suplee,...,TV-MA,1.0,NaN,7.5,11253.0,NaN,NaN,https://m.media-amazon.com/images/M/MV5BOTc3MD...,NaN,NaN
2,Red Widow,Mason & Sacramento St,NaN,Beyond Pix,American Broadcasting Company (ABC),Alon Aranya,Melissa Rosenberg,Radha Mitchell,Sterling Beaumon,Clifton Collins Jr.,...,TV-PG,60.0,NaN,6.5,3211.0,NaN,NaN,https://m.media-amazon.com/images/M/MV5BODMwND...,NaN,NaN
3,Shuang cheng gu shi,Pier 5,NaN,Envision Productions Inc.,Netflix,Tien-Lun Yeh,"Ling-Hui Chen, Nancy Chen, Chih-Chi Fan, Chia-...",Tammy Cheng,Peggy Tseng,Shen-Hao Wen,...,NaN,NaN,7 nominations,6.4,183.0,NaN,NaN,https://m.media-amazon.com/images/M/MV5BOTYwNj...,NaN,NaN
4,Americana,Palace of Fine Arts,NaN,Sutro Films LLC,NaN,Zachary Shedd,Zachary Shedd,Kelli Garner,Jack Davenport,Peter Coyote,...,R,81.0,1 win & 1 nomination,5.5,62.0,NaN,NaN,https://m.media-amazon.com/images/M/MV5BYjYxNG...,NaN,NaN


In [23]:
# Keep one row per film title so repeated filming locations do not over-count a movie.
# This is the only preparation step needed by the map cells below.
df_unique = df.drop_duplicates(subset=["Title"]).reset_index(drop=True)

print(f"Original rows: {len(df):,}")
print(f"Unique film/show titles: {len(df_unique):,}")

Original rows: 2,244
Unique film/show titles: 316


In [24]:
# Shared map settings and helper functions
SF_CENTER = [37.7749, -122.4194]
DARK_TILES = "CartoDB dark_matter"
HEAT_GRADIENT = {
    0.2: "#0000ff",
    0.4: "#00ffff",
    0.6: "#ffff00",
    0.8: "#ff7700",
    1.0: "#ff0000",
}


def make_base_map(zoom_start=12):
    """Create a consistent San Francisco base map."""
    return folium.Map(location=SF_CENTER, zoom_start=zoom_start, tiles=DARK_TILES)


def save_map(map_object, filename):
    """Save a Folium map to maps_output/ and return the path."""
    output_path = OUTPUT_DIR / filename
    map_object.save(output_path)
    print(f"✅ Saved: {output_path}")
    return output_path


def add_highlight_marker(map_object, coords, label, radius=15):
    """Add the gold neighborhood marker used in the story maps."""
    folium.CircleMarker(
        location=coords,
        radius=radius,
        popup=label,
        color="white",
        fill=True,
        fillColor="gold",
        fillOpacity=0.7,
        weight=3,
    ).add_to(map_object)
    return map_object

#### Filming location density map

This map shows where San Francisco filming locations are concentrated.

In [25]:
# Prepare location data
location_data = df_unique[["Latitude", "Longitude"]].dropna()
heat_coords = location_data[["Latitude", "Longitude"]].values.tolist()

print(f"Locations for heat map: {len(location_data):,}")
print(f"Latitude range: {location_data['Latitude'].min():.4f} to {location_data['Latitude'].max():.4f}")
print(f"Longitude range: {location_data['Longitude'].min():.4f} to {location_data['Longitude'].max():.4f}")

# Create map
filming_density_map = make_base_map()
HeatMap(
    heat_coords,
    radius=25,
    blur=15,
    max_zoom=1,
    gradient=HEAT_GRADIENT,
    min_opacity=0.3,
).add_to(filming_density_map)

save_map(filming_density_map, "filming_density_heatmap.html")
filming_density_map

Locations for heat map: 259
Latitude range: 37.7074 to 37.8452
Longitude range: -122.5301 to -122.3649
✅ Saved: maps_output/filming_density_heatmap.html


#### Box office heat map

This map weights filming locations by box office revenue. Warmer areas indicate locations associated with higher box office values.

In [26]:
# Prepare box office data
boxoffice_data = df_unique.dropna(subset=["Box_office", "Latitude", "Longitude"]).copy()

# Clean box office values such as "$123,456" into numbers
boxoffice_data["Box_office_clean"] = (
    boxoffice_data["Box_office"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
boxoffice_data["Box_office_numeric"] = pd.to_numeric(boxoffice_data["Box_office_clean"], errors="coerce")
boxoffice_data = boxoffice_data.dropna(subset=["Box_office_numeric"])

bo_min = boxoffice_data["Box_office_numeric"].min()
bo_max = boxoffice_data["Box_office_numeric"].max()

if bo_max == bo_min:
    boxoffice_data["bo_normalized"] = 1.0
else:
    boxoffice_data["bo_normalized"] = (
        (boxoffice_data["Box_office_numeric"] - bo_min) / (bo_max - bo_min)
    )

heat_data_boxoffice = boxoffice_data[["Latitude", "Longitude", "bo_normalized"]].values.tolist()

print(f"Box office data points: {len(boxoffice_data):,}")
print(f"Box office range: ${bo_min:,.0f} to ${bo_max:,.0f}")

# Create map
box_office_map = make_base_map()
HeatMap(
    heat_data_boxoffice,
    radius=25,
    blur=15,
    max_zoom=1,
    gradient=HEAT_GRADIENT,
    min_opacity=0.3,
).add_to(box_office_map)

save_map(box_office_map, "box_office_heatmap.html")
box_office_map

Box office data points: 163
Box office range: $1,802 to $330,455,270
✅ Saved: maps_output/box_office_heatmap.html


#### IMDb rating heat map

This map weights filming locations by IMDb rating. Warmer areas indicate locations associated with higher rated films.

In [27]:
# Prepare IMDb data
imdb_data = df_unique.dropna(subset=["Imdb_rating", "Latitude", "Longitude"]).copy()
imdb_data["Imdb_rating_numeric"] = pd.to_numeric(imdb_data["Imdb_rating"], errors="coerce")
imdb_data = imdb_data.dropna(subset=["Imdb_rating_numeric"])

# IMDb ratings are on a 0–10 scale, so divide by 10 for HeatMap intensity.
imdb_data["imdb_normalized"] = imdb_data["Imdb_rating_numeric"] / 10
heat_data_imdb = imdb_data[["Latitude", "Longitude", "imdb_normalized"]].values.tolist()

print(f"IMDb rating data points: {len(imdb_data):,}")
print(
    f"IMDb rating range: {imdb_data['Imdb_rating_numeric'].min():.1f} "
    f"to {imdb_data['Imdb_rating_numeric'].max():.1f}"
)

# Create map
imdb_rating_map = make_base_map()
HeatMap(
    heat_data_imdb,
    radius=25,
    blur=15,
    max_zoom=1,
    gradient=HEAT_GRADIENT,
    min_opacity=0.3,
).add_to(imdb_rating_map)

save_map(imdb_rating_map, "imdb_rating_heatmap.html")
imdb_rating_map

IMDb rating data points: 254
IMDb rating range: 3.8 to 8.8
✅ Saved: maps_output/imdb_rating_heatmap.html


#### Oscar recognition map

This map marks filming locations connected to films whose awards text includes “Oscar”.

In [28]:
# Prepare Oscar data
oscar_data = df_unique.dropna(subset=["Latitude", "Longitude"]).copy()
oscar_data["won_oscar"] = (
    oscar_data["Awards"]
    .fillna("")
    .astype(str)
    .str.contains("Oscar", case=False, regex=False)
)
oscar_winners = oscar_data[oscar_data["won_oscar"]].copy()

print(f"Total films with location data: {len(oscar_data):,}")
print(f"Oscar-winning films with coordinates: {len(oscar_winners):,}")

# Create map
oscar_winners_map = make_base_map()
for _, row in oscar_winners.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=6,
        popup=row.get("Title", "Oscar Winner"),
        color="white",
        fill=True,
        fillColor="gold",
        fillOpacity=0.7,
        weight=2,
    ).add_to(oscar_winners_map)

save_map(oscar_winners_map, "oscar_winners_map.html")
oscar_winners_map

Total films with location data: 259
Oscar-winning films with coordinates: 57
✅ Saved: maps_output/oscar_winners_map.html


#### Analysis behind the key neighborhood highlight maps

Before creating the single marker highlight maps, this section calculates why each neighborhood was selected:

- **Financial District/South Beach**: selected because it has the highest number of unique films / filming activity.
- **Lone Mountain/USF**: selected because it has the strongest combined score across box office and IMDb measures, driven especially by box office performance.
- **Lincoln Park**: selected because it has the highest average IMDb rating.
- **Haight Ashbury**: selected because it shows a useful contrast: high box office performance but relatively low IMDb rating.


In [29]:
# Analysis for choosing the highlighted neighborhoods
# This cell recreates the neighborhood-level success metrics used to justify the story maps.

NEIGHBORHOOD_COL = "Analysis neighborhood"

# 1) Filming activity: count unique titles per neighborhood
neighborhood_activity = (
    df_unique
    .dropna(subset=[NEIGHBORHOOD_COL])
    .groupby(NEIGHBORHOOD_COL)
    .agg(
        Film_Count=("Title", "nunique"),
        Avg_Latitude=("Latitude", "mean"),
        Avg_Longitude=("Longitude", "mean"),
    )
    .sort_values("Film_Count", ascending=False)
)

print("MOST FILMED NEIGHBORHOODS")
print("=" * 70)
display(neighborhood_activity.head(10))

# 2) Box office success: clean Box_office and calculate average revenue per neighborhood
boxoffice_by_neighborhood = boxoffice_data.groupby(NEIGHBORHOOD_COL).agg(
    Box_Office_Count=("Title", "nunique"),
    Avg_Box_Office=("Box_office_numeric", "mean"),
    Median_Box_Office=("Box_office_numeric", "median"),
    Total_Box_Office=("Box_office_numeric", "sum"),
).round(0)

boxoffice_by_neighborhood = boxoffice_by_neighborhood.sort_values("Avg_Box_Office", ascending=False)

print("\nBOX OFFICE SUCCESS BY NEIGHBORHOOD")
print("=" * 70)
display(boxoffice_by_neighborhood.head(10))

# 3) IMDb success: calculate average rating per neighborhood
imdb_by_neighborhood = imdb_data.groupby(NEIGHBORHOOD_COL).agg(
    IMDB_Count=("Title", "nunique"),
    Avg_IMDB_Rating=("Imdb_rating_numeric", "mean"),
    Median_IMDB_Rating=("Imdb_rating_numeric", "median"),
).round(2)

imdb_by_neighborhood = imdb_by_neighborhood.sort_values("Avg_IMDB_Rating", ascending=False)

print("\nIMDb RATING SUCCESS BY NEIGHBORHOOD")
print("=" * 70)
display(imdb_by_neighborhood.head(10))

# 4) Compare financial success and audience rating on a common 0–100 scale
neighborhood_success = (
    neighborhood_activity[["Film_Count"]]
    .join(boxoffice_by_neighborhood, how="left")
    .join(imdb_by_neighborhood, how="left")
    .fillna({
        "Box_Office_Count": 0,
        "Avg_Box_Office": 0,
        "Median_Box_Office": 0,
        "Total_Box_Office": 0,
        "IMDB_Count": 0,
        "Avg_IMDB_Rating": 0,
        "Median_IMDB_Rating": 0,
    })
)

max_avg_boxoffice = neighborhood_success["Avg_Box_Office"].max()
neighborhood_success["Box_Office_Score"] = (
    neighborhood_success["Avg_Box_Office"] / max_avg_boxoffice * 100
    if max_avg_boxoffice > 0 else 0
).round(1)

# IMDb is already on a 0–10 scale, so multiply by 10 to convert to 0–100.
neighborhood_success["IMDB_Score"] = (neighborhood_success["Avg_IMDB_Rating"] * 10).round(1)
neighborhood_success["Combined_Score"] = (
    (neighborhood_success["Box_Office_Score"] + neighborhood_success["IMDB_Score"]) / 2
).round(1)

neighborhood_success = neighborhood_success.sort_values("Combined_Score", ascending=False)

print("\nCOMPARATIVE SUCCESS SCORE BY NEIGHBORHOOD")
print("=" * 70)
display(neighborhood_success.head(15))

# 5) Pull out the neighborhoods used in the highlight maps and explain why they were chosen.
# The names below should match the neighborhood labels in the dataset.
selected_neighborhoods = [
    "Financial District/South Beach",
    "Lone Mountain/USF",
    "Lincoln Park",
    "Haight Ashbury",
]

key_neighborhood_analysis = neighborhood_success.loc[
    neighborhood_success.index.intersection(selected_neighborhoods)
].copy()

key_neighborhood_analysis["Why highlighted"] = [
    "Most filmed area / activity hub" if name == "Financial District/South Beach"
    else "Strongest combined success score, especially box office" if name == "Lone Mountain/USF"
    else "Highest average IMDb rating" if name == "Lincoln Park"
    else "High box office but lower IMDb rating; useful contrast case" if name == "Haight Ashbury"
    else "Selected narrative example"
    for name in key_neighborhood_analysis.index
]

print("\nKEY NEIGHBORHOODS USED FOR STORY HIGHLIGHT MAPS")
print("=" * 70)
display(key_neighborhood_analysis[
    [
        "Why highlighted",
        "Film_Count",
        "Avg_Box_Office",
        "Avg_IMDB_Rating",
        "Box_Office_Score",
        "IMDB_Score",
        "Combined_Score",
    ]
])


MOST FILMED NEIGHBORHOODS


,Film_Count,Avg_Latitude,Avg_Longitude
Analysis neighborhood,,,
Financial District/South Beach,34,37.791846,-122.400066
North Beach,26,37.802052,-122.407549
Nob Hill,24,37.791539,-122.412445
Tenderloin,19,37.783115,-122.414856
Chinatown,17,37.795749,-122.406374
Mission,14,37.759937,-122.416545
Treasure Island,13,37.820695,-122.369898
Russian Hill,13,37.803165,-122.419633
Marina,11,37.802168,-122.442562



BOX OFFICE SUCCESS BY NEIGHBORHOOD


,Box_Office_Count,Avg_Box_Office,Median_Box_Office,Total_Box_Office
Analysis neighborhood,,,,
Lone Mountain/USF,1,115000000.0,115000000.0,115000000
Excelsior,1,108248956.0,108248956.0,108248956
Haight Ashbury,2,107450007.0,107450007.0,214900014
Marina,7,94756901.0,32746941.0,663298308
Noe Valley,2,92302386.0,92302386.0,184604771
Golden Gate Park,3,91445801.0,79707906.0,274337403
Treasure Island,11,80639553.0,66308518.0,887035086
Russian Hill,8,79699014.0,43954470.0,637592110
Chinatown,9,71318027.0,35988495.0,641862241



IMDb RATING SUCCESS BY NEIGHBORHOOD


,IMDB_Count,Avg_IMDB_Rating,Median_IMDB_Rating
Analysis neighborhood,,,
Lincoln Park,2,7.85,7.85
Lone Mountain/USF,1,7.40,7.40
Lakeshore,1,7.40,7.40
South of Market,5,7.22,7.20
Golden Gate Park,6,7.08,7.20
Noe Valley,3,7.03,6.80
Castro/Upper Market,9,6.98,7.00
Treasure Island,13,6.98,6.90
Bernal Heights,2,6.95,6.95



COMPARATIVE SUCCESS SCORE BY NEIGHBORHOOD


,Film_Count,Box_Office_Count,Avg_Box_Office,Median_Box_Office,Total_Box_Office,IMDB_Count,Avg_IMDB_Rating,Median_IMDB_Rating,Box_Office_Score,IMDB_Score,Combined_Score
Analysis neighborhood,,,,,,,,,,,
Lone Mountain/USF,1,1.0,115000000.0,115000000.0,1.150000e+08,1,7.40,7.40,100.0,74.0,87.0
Excelsior,2,1.0,108248956.0,108248956.0,1.082490e+08,2,6.05,6.05,94.1,60.5,77.3
Noe Valley,3,2.0,92302386.0,92302386.0,1.846048e+08,3,7.03,6.80,80.3,70.3,75.3
Golden Gate Park,6,3.0,91445801.0,79707906.0,2.743374e+08,6,7.08,7.20,79.5,70.8,75.2
Haight Ashbury,2,2.0,107450007.0,107450007.0,2.149000e+08,2,5.65,5.65,93.4,56.5,75.0
Marina,11,7.0,94756901.0,32746941.0,6.632983e+08,11,6.65,6.40,82.4,66.5,74.4
Treasure Island,13,11.0,80639553.0,66308518.0,8.870351e+08,13,6.98,6.90,70.1,69.8,69.9
Russian Hill,13,8.0,79699014.0,43954470.0,6.375921e+08,12,6.29,6.35,69.3,62.9,66.1
Chinatown,17,9.0,71318027.0,35988495.0,6.418622e+08,17,6.82,7.10,62.0,68.2,65.1



KEY NEIGHBORHOODS USED FOR STORY HIGHLIGHT MAPS


,Why highlighted,Film_Count,Avg_Box_Office,Avg_IMDB_Rating,Box_Office_Score,IMDB_Score,Combined_Score
Analysis neighborhood,,,,,,,
Lone Mountain/USF,"Strongest combined success score, especially b...",1,115000000.0,7.40,100.0,74.0,87.0
Haight Ashbury,High box office but lower IMDb rating; useful ...,2,107450007.0,5.65,93.4,56.5,75.0
Financial District/South Beach,Most filmed area / activity hub,34,54882041.0,6.52,47.7,65.2,56.4
Lincoln Park,Highest average IMDb rating,2,19852304.0,7.85,17.3,78.5,47.9


#### Highlight maps for key neighborhoods

These maps create the single-marker neighborhood views used in the story.

In [30]:
# Coordinates for the key neighborhoods discussed in the narrative
highlight_neighborhoods = {
    "Financial District/South Beach": {
        "coords": [37.7946, -122.3999],
        "filename": "financial_district_map.html",
        "label": "Financial District/South Beach — most filmed area",
    },
    "Lone Mountain": {
        "coords": [37.7801, -122.4532],
        "filename": "lone_mountain_map.html",
        "label": "Lone Mountain",
    },
    "Lincoln Park": {
        "coords": [37.7836, -122.5029],
        "filename": "lincoln_park_map.html",
        "label": "Lincoln Park",
    },
    "Haight Ashbury": {
        "coords": [37.7699, -122.4469],
        "filename": "haight_ashbury_map.html",
        "label": "Haight Ashbury",
    },
}

highlight_maps = {}
for name, info in highlight_neighborhoods.items():
    neighborhood_map = make_base_map()
    add_highlight_marker(neighborhood_map, info["coords"], info["label"])
    save_map(neighborhood_map, info["filename"])
    highlight_maps[name] = neighborhood_map

# Display one example. Change the key below to display another highlight map.
highlight_maps["Financial District/South Beach"]

✅ Saved: maps_output/financial_district_map.html
✅ Saved: maps_output/lone_mountain_map.html
✅ Saved: maps_output/lincoln_park_map.html
✅ Saved: maps_output/haight_ashbury_map.html


## Discussion

## Contributions

We have started by brainstorming together and setting up the structure of the website.
Then we have split the analysis in the following way:
* Introduction - Hinojosa, Omar González (s) & Babeii, Denisa (s253528)
* 'San Francisco on screen' - Hinojosa, Omar González (s)
* Maps section - Babeii, Denisa (s253528)
* 'Looking beyond location' - Ilves, Ursula (s) - analysis, Hinojosa, Omar González (s) - infographics & text, Babeii, Denisa (s253528) - text
* 'What does this tell us?' - Hinojosa, Omar González (s) & Babeii, Denisa (s253528)
* 'Now it's your turn!' - Hinojosa, Omar González (s)
* Notebook parts 1-3 - Hinojosa, Omar González (s)
* Notebook part 4 - Babeii, Denisa (s253528)
* Notebook part 5 - Ilves, Ursula (s), Hinojosa, Omar González (s), Babeii, Denisa (s253528)
* Notebook part 6 - Ilves, Ursula (s)